In [19]:
import os
import pandas as pd
import numpy as np
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
df = pd.read_csv(path)

df.drop(columns=['catv_regroup','equipements','age','col','trajet','age', 'atm',
                 'sexe','catu','place','motor','choc','obsm','manv','jour','mois','an','hrmn'], inplace=True)

In [22]:
###############################################################
#   ENRICHISSEMENT « ROUTE LA PLUS PROCHE »  –  FRANCE métro
#   -> 1°×1° tiles   -> stockage Parquet   -> jointure batched
###############################################################
import time, requests, warnings
from pathlib import Path, PurePath
import pandas as pd, geopandas as gpd, numpy as np
from shapely.geometry import Point, shape

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------
# 0)  TON DATAFRAME df  (lat / long)  -------------------------------
# ----------------------------------------------------------------------
# df = pd.read_csv("accidents.csv")  # si besoin
assert {"lat", "long"}.issubset(df.columns), "df doit contenir lat & long"

# ----------------------------------------------------------------------
# 1)  GDF des accidents  (France métropolitaine) -----------------------
# ----------------------------------------------------------------------
df_fr = df[df.lat.between(41, 51) & df.long.between(-5, 9)].copy()
gdf_pts = gpd.GeoDataFrame(
    df_fr,
    geometry=[Point(xy) for xy in zip(df_fr.long, df_fr.lat)],
    crs="EPSG:4326",
)
print(f"✅ accidents retenus : {len(gdf_pts):,}")

# ----------------------------------------------------------------------
# 2)  Tuiles 1°×1°  ----------------------------------------------------
# ----------------------------------------------------------------------
tiles = [(lat, lon) for lon in range(-5, 10) for lat in range(41, 52)]
print(f"👉 tuiles à traiter : {len(tiles)}")

# ----------------------------------------------------------------------
# 3)  Tags utiles  (on démarre juste avec “highway”) -------------------
# ----------------------------------------------------------------------
TAGS_WAYS = {"highway": True}          # ➜ ajoute 'surface', 'maxspeed', etc. plus tard

# ----------------------------------------------------------------------
# 4)  Fonction téléchargement + sauvegarde Parquet ---------------------
# ----------------------------------------------------------------------
def download_tile(lat: int, lon: int, tags: dict, fn: str, sleep=6):
    """
    Télécharge la tuile (lat, lon) si absente.
    Enregistre un Parquet *seulement* si la GeoDataFrame est valide.
    """
    if Path(fn).exists():
        return fn                     # déjà présente

    bbox = f"{lat},{lon},{lat+1},{lon+1}"     # S,W,N,E
    query = (
        "[out:json][timeout:40];("
        + "".join([f'way["{k}"]({bbox});' for k in tags])
        + ");out geom;"
    )

    try:
        r = requests.get("https://overpass-api.de/api/interpreter",
                         params={"data": query}, timeout=60)
        r.raise_for_status()
        data = r.json()

        feats = []
        for elem in data.get("elements", []):
            if elem["type"] == "way" and "geometry" in elem:
                coords = [(n["lon"], n["lat"]) for n in elem["geometry"]]
                if len(coords) >= 2:
                    feats.append({
                        **{k: elem.get("tags", {}).get(k) for k in tags},
                        "geometry": shape({"type": "LineString",
                                           "coordinates": coords})
                    })

        gdf = gpd.GeoDataFrame(feats, geometry="geometry", crs="EPSG:4326")
        if not gdf.empty and "geometry" in gdf.columns:
            gdf.to_parquet(fn)
            print(f"✓ {PurePath(fn).name:<14} ({len(gdf):,} tronçons)")
        else:
            print(f"⏭ tuile vide {lat},{lon}")
    except Exception as e:
        print(f"❌ tuile ({lat},{lon})  →  {e}")
    finally:
        time.sleep(sleep)              # ~600 requêtes/h maxi Overpass
    return fn

# ----------------------------------------------------------------------
# 5)  Boucle de téléchargement  ----------------------------------------
# ----------------------------------------------------------------------
all_files = []
for lat, lon in tiles:
    fn = f"ways_{lat}_{lon}.parquet"
    if download_tile(lat, lon, TAGS_WAYS, fn) and Path(fn).exists():
        all_files.append(fn)

# ----------------------------------------------------------------------
# 6)  Concatène uniquement les Parquet valides -------------------------
# ----------------------------------------------------------------------
valid_files = []
for f in all_files:
    try:
        df_chk = pd.read_parquet(f)
        if "geometry" in df_chk.columns and df_chk["geometry"].notna().any():
            valid_files.append(f)
        else:
            print(f"⚠️ {PurePath(f).name} ignoré (pas de géométrie)")
    except Exception as e:
        print(f"⚠️ {PurePath(f).name} ignoré ({e})")

if not valid_files:
    raise RuntimeError("❌ Aucun fichier valide après contrôle.")

frames = []
for f in valid_files:
    df_tile = pd.read_parquet(f)
    if isinstance(df_tile.geometry.iloc[0], bytes):  # si la géométrie est du type binaire
        df_tile["geometry"] = gpd.GeoSeries.from_wkb(df_tile["geometry"])
    frames.append(df_tile)

gdf_ways = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
).drop_duplicates("geometry")

print(f"✅ tronçons OSM totaux : {len(gdf_ways):,}")

# ----------------------------------------------------------------------
# 7)  Jointure « plus proche »  (batch 100 000) -------------------------
# ----------------------------------------------------------------------
batch_size = 100_000
joined_parts = []
for i in range(0, len(gdf_pts), batch_size):
    sub = gdf_pts.iloc[i:i + batch_size]
    joined = gpd.sjoin_nearest(
        sub, gdf_ways,
        how="left",
        distance_col="dist_road",
        max_distance=100 / 111_000,   # 100 m → degrés
    ).drop(columns="index_right")
    joined_parts.append(joined)
    print(f"🟢 batch {i//batch_size+1} / {int(np.ceil(len(gdf_pts)/batch_size))}")

gdf_final = gpd.GeoDataFrame(pd.concat(joined_parts), crs="EPSG:4326")
final_df = gdf_final.drop(columns="geometry")

print("\n🎉 Aperçu du DataFrame enrichi :")
print(final_df.head())

# ----------------------------------------------------------------------
# 8)  Export optionnel
# ----------------------------------------------------------------------
# final_df.to_parquet("accidents_enrichis.parquet")

✅ accidents retenus : 580,067
👉 tuiles à traiter : 165
❌ tuile (44,-5)  →  Unknown column geometry
❌ tuile (45,-5)  →  Unknown column geometry
❌ tuile (46,-5)  →  Unknown column geometry
❌ tuile (49,-5)  →  Unknown column geometry
❌ tuile (44,-4)  →  Unknown column geometry
❌ tuile (45,-4)  →  Unknown column geometry
❌ tuile (46,-4)  →  Unknown column geometry
❌ tuile (49,-4)  →  Unknown column geometry
❌ tuile (44,-3)  →  Unknown column geometry
❌ tuile (45,-3)  →  Unknown column geometry
❌ tuile (41,4)  →  Unknown column geometry
❌ tuile (42,4)  →  Unknown column geometry
❌ tuile (41,5)  →  Unknown column geometry
❌ tuile (42,5)  →  Unknown column geometry
❌ tuile (41,6)  →  Unknown column geometry
❌ tuile (41,7)  →  Unknown column geometry
❌ tuile (42,7)  →  Unknown column geometry
✅ tronçons OSM totaux : 29,297,514
🟢 batch 1 / 6
🟢 batch 2 / 6
🟢 batch 3 / 6
🟢 batch 4 / 6
🟢 batch 5 / 6
🟢 batch 6 / 6

🎉 Aperçu du DataFrame enrichi :
   obs  grav  lum dep    com  agg  int        lat   

In [23]:
final_df.to_csv("../data/processed/accidents_enriched.csv", index=False)